In [11]:
import pandas as pd, json
from pathlib import Path

In [12]:
github_csv = pd.read_csv("projects.csv", dtype=str).fillna("")
github_csv["name_norm"] = github_csv["name"].str.lower().str.strip()

github_projects = [
    {
        "name": r["name"],
        "name_norm": r["name_norm"],
        "visibility": r.get("visibility", ""),
        "language": r.get("language", ""),
        "license": r.get("license", ""),
        "updated": r.get("updatedAt", "")
    }
    for _, r in github_csv.iterrows()
]

github_map = {p["name_norm"]: p for p in github_projects}


data = json.load(open("projects.json", "r", encoding="utf-8"))
prime = data.get("prime", {})

registry_projects = []
for category, items in prime.items():
    for it in items:
        ent = dict(it)
        ent["category"] = category
        ent["name_norm"] = ent.get("name", "").lower().strip()
        registry_projects.append(ent)

registry_map = {p["name_norm"]: p for p in registry_projects}

In [13]:
rows = []
registry_names = set(registry_map.keys())
github_names = set(github_map.keys())

for name_norm, reg in registry_map.items():
    gh = github_map.get(name_norm)  

    rows.append({
        "updated": gh["updated"] if gh else "",
        "category": reg.get("category"),
        "registry_name": reg.get("name"),
        "language": gh["language"] if gh else "",
        "status": reg.get("status", ""),

        "visibility": gh["visibility"] if gh else "",
        "github_name": gh["name"] if gh else "",
        "github_present": gh is not None,
    })

df = (
    pd.DataFrame(rows)
    .sort_values(["category", "registry_name"])
    .reset_index(drop=True)
)

matched = df[df["github_present"]]
only_registry = df[~df["github_present"]]
only_github = sorted(github_names - registry_names)

print(f"Registry JSON total: {len(registry_projects)}")
print(f"GitHub CSV total: {len(github_projects)}")
print(f"Matched (in both): {len(matched)}")
print(f"Only in registry JSON: {len(only_registry)}")
print(f"Only in GitHub CSV: {len(only_github)}")

Registry JSON total: 50
GitHub CSV total: 50
Matched (in both): 50
Only in registry JSON: 0
Only in GitHub CSV: 0


In [14]:
if only_github:
    print("Projects on GitHub not found in Registry:")
    display(pd.DataFrame({"name_norm": only_github}))


if not only_registry.empty:
    print("Projects in Registry but missing on GitHub:")
    display(only_registry[["registry_name", "category"]])

In [15]:
display(df)

,updated,category,registry_name,language,status,visibility,github_name,github_present
0,2026-03-06 12:48:54,*A作品,H,,live,PUBLIC,H,True
1,2026-03-01 00:34:38,*A作品,K,,live,PUBLIC,K,True
2,2026-03-10 08:50:04,*A作品,W,,archived,PRIVATE,W,True
3,2026-03-04 08:43:51,*Personal,.games,,live,PUBLIC,.games,True
4,2026-03-01 10:20:52,*Personal,.info,,live,PRIVATE,.info,True
5,2026-03-01 01:13:08,*Personal,.notes,,live,PRIVATE,.notes,True
6,2026-03-01 10:04:23,*Personal,.terminal,,live,PRIVATE,.terminal,True
7,2026-03-08 07:44:59,*Personal,probabilityzero,,live,PUBLIC,probabilityzero,True
8,2025-03-11 09:41:15,*Personal,yearofthesnake,,live,PUBLIC,yearofthesnake,True
9,2026-02-28 07:02:28,Applications,charview,,archived,PRIVATE,charview,True
